In [88]:
import pandas as pd
import numpy as np

movies_df = pd.read_csv('data/movies.csv')
movies = movies_df.copy()

ratings_df = pd.read_csv('data/ratings.csv')
ratings = ratings_df.copy()

#tags_df = pd.read_csv('data/tags.csv')
#tags = tags_df.copy()

Filtering ratings. Bad movies and impolular movies are excluded

In [154]:
lowest_acceptable_mean_rating_score = 1.5 # Set to zero to disable filtering
lowest_acceptable_rating_count = 50 # Set to zero to disable filtering

ratings_grouped = ratings.groupby('movieId')['rating'].mean().sort_values()
good_movie_scores = ratings_grouped[ratings_grouped >= lowest_acceptable_mean_rating_score].index
ratings_good = ratings[ratings["movieId"].isin(good_movie_scores)]

ratings_per_movie = ratings.groupby('movieId')['rating'].count()
popular_movie = ratings_per_movie[ratings_per_movie >= lowest_acceptable_rating_count].index
ratings_good_and_popular = ratings_good[ratings_good["movieId"].isin(popular_movie)]


Splitting movie genres string with on-hot encoder and excluding movies without genre

In [155]:
genre_dummies = movies["genres"].str.get_dummies("|")
movies_genre_one_hot = pd.concat([movies, genre_dummies], axis=1)
movies_filtered = movies_genre_one_hot[movies_genre_one_hot["(no genres listed)"] == 0]
movies_filtered = movies_filtered.drop(columns= ["genres", "(no genres listed)"])




Filtering bad and impopular movies from movies

In [156]:
ratings_good_and_popular_list = ratings_good_and_popular["movieId"].unique()
movies_filtered = movies_filtered[movies_filtered["movieId"].isin(ratings_good_and_popular_list)]

movies_filtered = movies_filtered.reset_index(drop=True)
all_movies_genre_matrix = movies_filtered.drop(columns=["movieId", "title"])



Merging filtered movies and ratings dataframe (ta bort?)

In [157]:
#movies_ratings_merge = ratings_filtered.merge(movies_filtered, on="movieId", how="inner")
#movies_ratings_merge.shape

Rekommender Contet
"identify similarities between movies based on genres based filtering KNN och cosine similarity"
"one-hot encoding on genres and a KNN-Transform with cosine similarity"

In [158]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_Cosine(movie_title, n=5):
    matches = movies_filtered[movies_filtered["title"] == movie_title]
    if matches.empty:
       return "Movie not found"
    
    idx = matches.index[0]
    all_movies_genre = movies_filtered.drop(columns=["movieId", "title"])
    input_movie_genre = all_movies_genre.iloc[idx].values.reshape(1, -1)

    # Inputfilm vs all movies -> Similarity
    similarity_scores = cosine_similarity(input_movie_genre, all_movies_genre).flatten()

    similarity_scores_df = pd.DataFrame({"similarity_score_%": 100*similarity_scores}) # Array from cosine_similarity to pandas dataframe
    similarity_scores_sorted_df = similarity_scores_df.sort_values(by="similarity_score_%", ascending=False)
    similarity_scores_sorted_top5_df = similarity_scores_sorted_df.iloc[1:n+1]   # Removing self
    similarity_df = similarity_scores_sorted_top5_df                             # shortening name

    similarity_df["title"] = movies_filtered.loc[similarity_df.index, "title"]  # Replaces movieId with title 
    similarity_df = similarity_df.loc[:, ['title', 'similarity_score_%']]         # Changing order of columns
    return similarity_df
recommend_Cosine("Yellow Submarine (1968)", 20)

,title,similarity_score_%
6890,"Chipmunk Adventure, The (1987)",91.287093
12089,Frozen (2013),91.287093
6410,Allegro non troppo (1977),89.442719
6957,Into the Woods (1991),89.442719
15547,Adventure Time: Islands (2017),89.442719
8086,"Thief and the Cobbler, The (a.k.a. Arabian Kni...",89.442719
15320,KonoSuba: God's Blessing on this Wonderful Wor...,89.442719
9116,How the Grinch Stole Christmas! (1966),89.442719
9872,Shrek the Halls (2007),89.442719
15890,Luck (2022),89.442719


In [159]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(all_movies_genre_matrix)

def recommend_KNN(movie_title, n=5):
    matches = movies_filtered[movies_filtered["title"] == movie_title]
    if matches.empty:
       return "Movie not found"

    idx = matches.index[0]
    input_movie_genre = all_movies_genre_matrix.iloc[[idx]]

    knn_distances, knn_indices = knn.kneighbors(input_movie_genre, n_neighbors=n+1)
    # platta arrays
    indices = knn_indices.flatten()[1:]      # hoppa över sig själv
    distances = knn_distances.flatten()[1:]

    result = movies_filtered.iloc[indices][["title"]].copy()

    similarity_scores = 100 - 100*distances    # cosine distance → similarity
    result["similarity_score_%"] = similarity_scores

    return result

recommend_KNN("Yellow Submarine (1968)", 20)


,title,similarity_score_%
6890,"Chipmunk Adventure, The (1987)",91.287093
12089,Frozen (2013),91.287093
15890,Luck (2022),89.442719
6410,Allegro non troppo (1977),89.442719
9116,How the Grinch Stole Christmas! (1966),89.442719
9872,Shrek the Halls (2007),89.442719
6957,Into the Woods (1991),89.442719
8086,"Thief and the Cobbler, The (a.k.a. Arabian Kni...",89.442719
15320,KonoSuba: God's Blessing on this Wonderful Wor...,89.442719
15935,Wendell & Wild (2022),89.442719
